In [1]:
import os
os.chdir('../')

In [9]:
import torch
from pathlib import Path

def load_feats(d, k="inception_feature"):
    xs=[]
    for p in Path(d).rglob("*.pt"):
        v = torch.load(p, map_location="cpu")
        v = v[k] if isinstance(v, dict) else v
        xs.append(v.view(-1, 2048).float())
    return torch.cat(xs, 0)  # (B, 2048)

@torch.no_grad()
def precision_recall_torch(ref_features: torch.Tensor,
                           sample_features: torch.Tensor,
                           k: int = 3,
                           normalize: bool = False) -> tuple[float, float]:
    """
    Kynkäänniemi et al. (NeurIPS'19) k-NN 매니폴드 방식 Precision/Recall.
    - ref_features : 실측(레퍼런스) 특징 [Nr, D] (e.g., Inception pool3 2048-d)
    - sample_features : 생성(샘플) 특징 [Ng, D]
    - k : 자기 집합 내 k번째 이웃 반경
    - normalize : 각 벡터 L2 정규화 후 L2 거리 사용 여부
    반환값: (precision, recall)
      * precision = 생성 샘플이 '실측 매니폴드' 안에 들어오는 비율
      * recall    = 실측 샘플이 '생성 매니폴드' 안에 들어오는 비율
    """
    assert ref_features.dim() == 2 and sample_features.dim() == 2
    assert ref_features.size(1) == sample_features.size(1), "feature dim mismatch"

    R = ref_features
    G = sample_features

    if normalize:
        R = torch.nn.functional.normalize(R, dim=1)
        G = torch.nn.functional.normalize(G, dim=1)

    def knn_radii(X: torch.Tensor, k: int) -> torch.Tensor:
        n = X.size(0)
        if n == 1:
            return torch.zeros(1, device=X.device, dtype=X.dtype)
        k_eff = max(1, min(k, n - 1))
        D = torch.cdist(X, X)                     # [n, n] Euclidean
        D.fill_diagonal_(float('inf'))            # 자기 자신 제외
        kth, _ = D.kthvalue(k_eff, dim=1)         # 각 행의 k번째 최소 거리
        return kth                                # [n], Euclidean 반경

    # 1) 각 집합 내 k-NN 반경
    r = knn_radii(R, k)   # [Nr]
    s = knn_radii(G, k)   # [Ng]

    # 2) 교차 거리 (실측 vs 생성)
    D_rg = torch.cdist(R, G)  # [Nr, Ng]

    # precision: ∀ g_j, ∃ r_i s.t. d(r_i, g_j) <= r_i
    precision = (D_rg <= r[:, None]).any(dim=0).float().mean().item()

    # recall: ∀ r_i, ∃ g_j s.t. d(r_i, g_j) <= s_j
    recall    = (D_rg <= s[None, :]).any(dim=1).float().mean().item()

    return precision, recall


In [12]:
valid_features = torch.load('/dataset/dit/valid_feats.pt')['features']
valid_features.shape

torch.Size([10000, 2048])

In [16]:
for i in range(0, 15):
    dir = f"samplings/GMDiT/1.4/3/Dual-Solver/10000/which_{i}/"
    if not os.path.exists(dir):
        continue

    sample_features = load_feats(dir)
    precision, recall = precision_recall_torch(valid_features, sample_features, normalize=True)
    print(dir, precision, recall)

samplings/GMDiT/1.4/3/Dual-Solver/10000/which_4/ 0.828499972820282 0.7027000188827515
samplings/GMDiT/1.4/3/Dual-Solver/10000/which_5/ 0.8514999747276306 0.7120000123977661
samplings/GMDiT/1.4/3/Dual-Solver/10000/which_6/ 0.8503999710083008 0.7139000296592712
samplings/GMDiT/1.4/3/Dual-Solver/10000/which_7/ 0.8554999828338623 0.7200000286102295
samplings/GMDiT/1.4/3/Dual-Solver/10000/which_9/ 0.8550000190734863 0.717199981212616
samplings/GMDiT/1.4/3/Dual-Solver/10000/which_10/ 0.8560000061988831 0.7164999842643738
samplings/GMDiT/1.4/3/Dual-Solver/10000/which_11/ 0.8526999950408936 0.7213000059127808
samplings/GMDiT/1.4/3/Dual-Solver/10000/which_12/ 0.8478999733924866 0.7178000211715698
samplings/GMDiT/1.4/3/Dual-Solver/10000/which_13/ 0.85589998960495 0.7231000065803528
